# PRACTICA INVESTIGATIVA II

### ESTUDIO DE REPLICACIÓN CONCEPTUAL DEL PAPER TITULADO: 
**To quantum or not to quantum: towards algorithm selection in near-term quantum optimization**

# 1. Introducción

# 2. Antecedentes

Instalación de las librerias

In [3]:
#%pip install networkx
#%pip install qiskit
#%pip install qiskit-optimization
#%pip install qiskit-algorithms
#%pip install cvxpy
#%pip install scipy
#%pip install scikit-learn

Importaciones de las librerias

In [11]:
from networkx import random_regular_graph
import networkx as nx
import networkx.algorithms.approximation as approx

import itertools

from qiskit_optimization.applications import Maxcut

from qiskit_algorithms import QAOA
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_algorithms.optimizers import NELDER_MEAD
from qiskit_optimization.algorithms import MinimumEigenOptimizer

import cvxpy as cp

import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


Generación del conjunto de grafos regulares de grado 4 aleatorios con 11 a 24 nodos, con 20 instancias por cantidad de nodos para un total de 280 grafos diferentes para las pruebas.

Esto generado mediante la libreria *networkx* y su método *random_regular_graph*

In [8]:
def generate_graphs(nMin,nMax,q):
    gs = []

    for n in range(nMin,nMax):
        for i in range(q):
            gs.append(random_regular_graph(4,n,i*10))
            gs.append(random_regular_graph(4,n,(i+1)*11))
    
    return gs

Algoritmo ingenuo

In [13]:
def brute_force_maxcut(G):
    nodes = list(G.nodes())
    n = len(nodes)
    
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    indexed_edges = [(node_to_idx[u], node_to_idx[v]) for u, v in G.edges()]

    best_cut_value = -1
    best_partition = None

    # Recorremos todas las posibles particiones binarias
    # Cada nodo puede estar en grupo 0 o grupo 1
    for rest_of_assignment in itertools.product([0, 1], repeat=n-1):
        cut_value = 0
        
        assignment = (0,) + rest_of_assignment
        
        for i, j in indexed_edges:
            if assignment[i] != assignment[j]:
                cut_value += 1

        if cut_value > best_cut_value:
            best_cut_value = cut_value
            best_partition = assignment 

   
    group_0 = [nodes[i] for i, val in enumerate(best_partition) if val == 0]
    group_1 = [nodes[i] for i, val in enumerate(best_partition) if val == 1]

    best_partition = (group_0, group_1)

    return best_cut_value, best_partition

Algoritmo cuantico QAOA

In [5]:
#Convertir MaxCut en un problema QUBO

def graph_to_qubo(G):
    maxcut = Maxcut(G)
    qp = maxcut.to_quadratic_program()
    return qp


#Resolver el problema de MaxCut con QAOA

def solve_qaoa(G):
    qp = graph_to_qubo(G)

    sampler = Sampler()
    optimizer = NELDER_MEAD(maxiter=50) #O COBYLA

    qaoa = QAOA(sampler=sampler, optimizer=optimizer, reps=1)

    solver = MinimumEigenOptimizer(qaoa)
    result = solver.solve(qp)

    return int(result.fval) , result.x  # valor del corte

Algoritmos clasico GW

In [6]:
def goemans_williamson_max_cut(G, num_roundings=100, weight_attr='weight'):
    n = G.number_of_nodes()
    if n == 0:
        return 0, ([], []), 0

    nodes_list = list(G.nodes)
    
    L = nx.laplacian_matrix(G, weight=weight_attr).toarray()
    
    X = cp.Variable((n, n), PSD=True)
    
    objective = cp.Maximize(0.25 * cp.trace(L @ X))
    
    constraints = [cp.diag(X) == 1]
    
    prob = cp.Problem(objective, constraints)
    prob.solve()
    
    sdp_upper_bound = prob.value
    X_val = X.value
    
    evals, evecs = np.linalg.eigh(X_val)
    evals[evals < 0] = 0 
    
    V = evecs @ np.diag(np.sqrt(evals))
    
    best_cut_value = -1
    best_partition = ([], [])
    
    for _ in range(num_roundings):
        r = np.random.randn(n)
        
        signs = np.sign(V @ r)
        
        S1 = [nodes_list[i] for i in range(n) if signs[i] >= 0]
        S2 = [nodes_list[i] for i in range(n) if signs[i] < 0]

        current_cut_value = nx.cut_size(G, S1, S2, weight=weight_attr)

        if current_cut_value > best_cut_value:
            best_cut_value = current_cut_value
            best_partition = (S1, S2)
            
    return best_cut_value, best_partition, sdp_upper_bound


Generación de pruebas iniciales para la correctitud de los algoritmos

In [15]:
results = []
analisis = []

gs = generate_graphs(6,7,1)

for G in gs:
    result_brute_force_maxcut, partition_brute_force_maxcut  =  brute_force_maxcut(G)
    result_QAOA, partition_qaoa = solve_qaoa(G)
    result_GW , partition_GW , _ = goemans_williamson_max_cut(G)
    
    results.append([result_brute_force_maxcut,result_QAOA,result_GW])
    analisis.append([partition_brute_force_maxcut,partition_qaoa,partition_GW ])
    
print(results)
print(analisis)

c:\Users\samue\OneDrive\Desktop\Practica investigativa\.venv\Lib\site-packages\scipy\sparse\linalg\_dsolve\linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
c:\Users\samue\OneDrive\Desktop\Practica investigativa\.venv\Lib\site-packages\scipy\sparse\linalg\_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
c:\Users\samue\OneDrive\Desktop\Practica investigativa\.venv\Lib\site-packages\scipy\sparse\_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])
c:\Users\samue\OneDrive\Desktop\Practica investigativa\.venv\Lib\site-packages\scipy\sparse\linalg\_dsolve\linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
c:\Users\samue\OneDrive\Desktop\Practica investigativa\.venv\Lib\site-packages

[[8, 8, 8], [8, 8, 8]]
[[([0, 1, 3, 4], [2, 5]), array([1., 0., 0., 1., 0., 0.]), ([2, 5], [0, 1, 3, 4])], [([0, 1, 2, 3], [4, 5]), array([1., 0., 0., 0., 1., 1.]), ([4, 5], [0, 1, 2, 3])]]


#  3. Rendimiento en grafos regulares de grado 4


# 4. Construcción del Modelo de ML de selección de algoritmo hibridos

## Extracción de los features

Features relacionadas al grafos regulare

In [ ]:
#Common features related to regular graphs and the spectrum of the Laplacian matrix

#Return density, log norm laplacian ev(1,2,3,4,5), log laplacian ev ratio, spectral gap

def extract_graph_features(G):
    n = G.number_of_nodes()
    
    if n == 0:
        return None

    density = nx.density(G)
    
    L_norm = nx.normalized_laplacian_matrix(G).toarray()
    
    evals_norm = np.linalg.eigvalsh(L_norm)

    evals_norm = np.maximum(evals_norm, 0)

    evals_norm = np.sort(evals_norm)[::-1]
    

    top_5_norm = np.zeros(5)
    for i in range(min(n, 5)):
        top_5_norm[i] = evals_norm[i]
        
    eps = 1e-10
    log_norm_evs = np.log(top_5_norm + eps)
    

    L_std = nx.laplacian_matrix(G).toarray()
    evals_std = np.linalg.eigvalsh(L_std)
    evals_std = np.maximum(evals_std, 0)
    

    evals_std_desc = np.sort(evals_std)[::-1]
    
    evals_std_asc = np.sort(evals_std)
    
    if n >= 2:
        ratio = (evals_std_desc[0] + eps) / (evals_std_desc[1] + eps)
        log_laplacian_ev_ratio = np.log(ratio)
    else:
        log_laplacian_ev_ratio = 0.0
        
    spectral_gap = evals_std_asc[1] if n >= 2 else 0.0

    features = {
        'density': density,
        'log_norm_laplacian_ev1': log_norm_evs[0],
        'log_norm_laplacian_ev2': log_norm_evs[1],
        'log_norm_laplacian_ev3': log_norm_evs[2],
        'log_norm_laplacian_ev4': log_norm_evs[3],
        'log_norm_laplacian_ev5': log_norm_evs[4],
        'log_laplacian_ev_ratio': log_laplacian_ev_ratio,
        'spectral_gap': spectral_gap
    }
    
    return features

In [ ]:
# Set numbers for graphs

#Return independence number over number edges, matching number over number edges, 
# diameter over number edges, domination number over number nodes, 
# zero forcing number over number nodes, power domination over number edges

def extract_set_number_features(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    
    if n == 0 or m == 0:
        return {
            'independence_number_over_number_edges': 0,
            'matching_number_over_number_edges': 0,
            'diameter_over_number_edges': 0,
            'domination_number_over_number_nodes': 0,
            'zero_forcing_number_over_number_nodes': 0,
            'power_domination_over_number_edges': 0
        }

    indep_num = len(approx.maximum_independent_set(G))
    
    matching_num = len(nx.maximal_matching(G))
    
    if nx.is_connected(G):
        diameter = nx.diameter(G)
    else:
        components = (G.subgraph(c) for c in nx.connected_components(G))
        diameter = max(nx.diameter(c) for c in components)
        
    dom_num = len(approx.min_weighted_dominating_set(G))

    def get_zero_forcing_number(G):
        n_nodes = G.number_of_nodes()
        min_zf = n_nodes
        nodes = list(G.nodes())
        for i in range(1, min(n_nodes, 4)):
            import itertools
            for subset in itertools.combinations(nodes, i):
                active = set(subset)
                while True:
                    new_active = False
                    current_active = list(active)
                    for v in current_active:
                        neighbors = set(G.neighbors(v))
                        inactive_neighbors = neighbors - active
                        if len(inactive_neighbors) == 1:
                            active.add(inactive_neighbors.pop())
                            new_active = True
                    if not new_active:
                        break
                if len(active) == n_nodes:
                    return i
        return min_zf

    zf_num = get_zero_forcing_number(G)
    
    # Power Domination approximation
    pd_num = dom_num # Proxy standard as exact is NP-Hard and specific
    
    return {
        'independence_number_over_number_edges': indep_num / m,
        'matching_number_over_number_edges': matching_num / m,
        'diameter_over_number_edges': diameter / m,
        'domination_number_over_number_nodes': dom_num / n,
        'zero_forcing_number_over_number_nodes': zf_num / n,
        'power_domination_over_number_edges': pd_num / n
    }

Features relacionadas al resultado de GW

In [ ]:
from scipy.linalg import cholesky

#Features related to the relaxed solution of the semi-definite program in GW and the random projection routine

# Return percent cut,  percent positive lower part relaxation solution, 
# percent close1 lower part relaxation solution, percent close lower part relaxation solution, 
# expected costGW over sdp cost
# std costGW over sdp cost

def extract_gw_relaxation_features(G, num_projections=1000):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    
    if n < 2 or m == 0:
        return {
            'percent_cut': 0.0,
            'percent_positive_lower_part_relaxation_solution': 0.0,
            'percent_close1_lower_part_relaxation_solution': 0.0,
            'percent_close3_lower_part_relaxation_solution': 0.0,
            'expected_costGW_over_sdp_cost': 0.0,
            'std_costGW_over_sdp_cost': 0.0
        }

    nodes_list = list(G.nodes)
    L = nx.laplacian_matrix(G).toarray()
    
    X = cp.Variable((n, n), PSD=True)
    objective = cp.Maximize(0.25 * cp.trace(L @ X))
    constraints = [cp.diag(X) == 1]
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.SCS) # SCS es rápido para estas dimensiones
    
    sdp_cost = prob.value
    X_val = X.value
    
    # Asegurar que la matriz sea definida positiva para Cholesky por errores numéricos
    X_val = (X_val + X_val.T) / 2
    X_val += np.eye(n) * 1e-9
    
    V = cholesky(X_val, lower=True)
    
    # Extraer elementos de la parte inferior (excluyendo diagonal si se prefiere, 
    # aquí tomamos la matriz triangular inferior completa resultante de Cholesky)
    lower_elements = V[np.tril_indices(n)]
    n_elems = lower_elements.size

    percent_positive = np.sum(lower_elements > 0) / n_elems
    percent_close1 = np.sum(np.abs(lower_elements) < 0.1) / n_elems
    percent_close3 = np.sum(np.abs(lower_elements) < 0.001) / n_elems
    
    costs = []
    for _ in range(num_projections):
        r = np.random.randn(n)
        signs = np.sign(V.T @ r) # V.T porque Cholesky lower da X = V V^T
        S1 = [nodes_list[i] for i in range(n) if signs[i] >= 0]
        S2 = [nodes_list[i] for i in range(n) if signs[i] < 0]
        costs.append(nx.cut_size(G, S1, S2))
        
    costs = np.array(costs)
    
    return {
        'percent_cut': sdp_cost / m,
        'percent_positive_lower_part_relaxation_solution': percent_positive,
        'percent_close1_lower_part_relaxation_solution': percent_close1,
        'percent_close3_lower_part_relaxation_solution': percent_close3,
        'expected_costGW_over_sdp_cost': np.mean(costs) / sdp_cost,
        'std_costGW_over_sdp_cost': np.std(costs) / sdp_cost
    }

Variables Globales de la generación de todas las features para los modelos

In [ ]:
class AllFeatures:
    def __init__(self,graphs):
        self.featuresI = []
        self.featuresII = []
        self.featuresIII = []
        self.featuresGW_std_and_expected_costGW = []
    
        self.qaoa_vals = []
        self.gw_vals = []
        
        self.graphs = graphs
        
        self.bf_vals = []
    
    
        for G in graphs:
            try:
                qaoa_value,_ = solve_qaoa(G)
            except:
                continue
        
            gw_value,_,_ = goemans_williamson_max_cut(G)
        
            self.qaoa_vals.append(qaoa_value)
            self.gw_vals.append(gw_value)

            gw_relaxation_features = extract_gw_relaxation_features(G)
        
            self.featuresI.append(list(extract_graph_features(G).values()))
            self.featuresII.append(list(extract_set_number_features(G).values()))
            self.featuresIII.append(list(gw_relaxation_features.values())) 
            self.featuresGW_std_and_expected_costGW.append([gw_relaxation_features["expected_costGW_over_sdp_cost"],gw_relaxation_features["std_costGW_over_sdp_cost"]])
    
    def set_data_brute_force(self):
        for G in self.graphs:
            self.bf_vals.append(brute_force_maxcut(G))

## Metodos para construir el conjunto de datos (data-set)

### Predicting Criterion (1): QAOA vs. GW

Definición de la creación del conjunto de datos tomando todas las features recomendadas en el paper

In [ ]:
def build_dataset_creterio_one_all_features(graphs,data):
    X = []
    y = []

    for i in range(len(graphs)):
        featuresI = data.featuresI[i]
        featuresII = data.featuresII[i]
        featuresIII = data.featuresIII[i]
        
        allFeatures = featuresI + featuresII + featuresIII

        qaoa_val = data.qaoa_vals[i]
       
        gw_val = data.gw_vals[i]
        
        label = 1 if gw_val > qaoa_val else 0

        X.append(allFeatures)
        y.append(label)

    return X, y

Definición de la creación del conjunto de datos tomando unicamente las features relacionadas al comportamiento del GW

In [ ]:
def build_dataset_creterio_one_gw_features(graphs,data):
    X = []
    y = []

    for i in range(len(graphs)):
        featuresGW = data.featuresGW_std_and_expected_costGW[i]

        qaoa_val = solve_qaoa(G)
       
        gw_val,_,_ = goemans_williamson_max_cut(G)

        label = 1 if gw_val > qaoa_val else 0

        X.append(featuresGW)
        y.append(label)

    return X, y

### Predicting Criterion (2): QAOA as a High-Performing Heuristic

Definición de la creación del conjunto de datos tomando todas las features recomendadas en el paper

In [ ]:
def build_dataset_creterio_two_all_features(graphs,data):
    X = []
    y = []

    for i in range(len(graphs)):
        featuresI = data.featuresI[i]
        featuresII = data.featuresII[i]
        featuresIII = data.featuresIII[i]
        
        allFeatures = featuresI + featuresII + featuresIII
        
        qaoa_val = data.qaoa_vals[i]
       
        gw_val = data.gw_vals[i]
        
        data.set_data_brute_force()
        
        max_cut_value = data.bf_vals[i]
        
        ratio_qaoa = qaoa_val / max_cut_value
        ratio_gw = gw_val / max_cut_value 
        
        label = 1 if qaoa_val >= max_cut_value * 0.98 and ratio_qaoa >= ratio_gw + 0.02 else 0 

        X.append(allFeatures)
        y.append(label)

    return X, y

Definición de la creación del conjunto de datos tomando solo las features basicas de un grafo

In [ ]:
def build_dataset_creterio_two_first_features(graphs,data):
    X = []
    y = []

    for i in range(len(graphs)):
        featuresI = data.featuresI[i]
        
        qaoa_val = data.qaoa_vals[i]
       
        gw_val = data.gw_vals[i]
        
        data.set_data_brute_force()
        
        max_cut_value = data.bf_vals[i]
        
        ratio_qaoa = qaoa_val / max_cut_value
        ratio_gw = gw_val / max_cut_value 
        
        label = 1 if qaoa_val >= max_cut_value * 0.98 and ratio_qaoa >= ratio_gw + 0.02 else 0 

        X.append(featuresI)
        y.append(label)

    return X, y

## Entrenamiento del Modelo con Sklearn

In [ ]:
def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    model = RandomForestClassifier()
    model.fit(X_train, y_train)

    acc = model.score(X_test, y_test)

    return model, acc

## Pipeline final

In [ ]:
#Constantes globales

NMIN = 11
NMAX = 25
Q = 10

In [ ]:
# Pipeline general

graphs = generate_graphs(NMIN,NMAX,Q)

data = AllFeatures(graphs)


def pipeline (build_dataset):

    X, y = build_dataset(graphs,data)

    model, acc = train_model(X, y)
    
    return model , acc


# Predicting Criterion (1): QAOA vs. GW

""" model_1_all_features , acc_1_all_features = pipeline(build_dataset_creterio_one_all_features) """

model_1_gw_features,  acc_1_gw_features = pipeline(build_dataset_creterio_one_gw_features)

# Predicting Criterion (2): QAOA as a High-Performing Heuristic

""" model_2_all_features,  acc_2_all_features = pipeline(build_dataset_creterio_two_all_features)

model_2_first_features ,  acc_2_first_features = pipeline(build_dataset_creterio_two_first_features) """


print(acc_1_gw_features)

## Validación cruzada de cada modelo